# Lab 3 — Hallucination (EP02)

Lab 2 checked that the **format** is right. But what if the model **makes up facts**?

**What you'll learn**
1. **consistency sampling**: ask 5 times and see whether the answers agree
2. **log-prob**: the average probability of the tokens in the answer (continuing from EP01)
3. the limit: both methods catch **uncertainty**, not **incorrectness**; we check it with code to see for ourselves
4. **grounding**: answer only from a document, with a `quote` our code can verify

The paper *Temporal Patch Attention for Thai Traffic Forecasting* and the course syllabus in this lab are **made up for teaching**;
every model answer comes from a real run. (The video asks in Thai; this English notebook asks in English, so your outputs may differ.)

> Reference: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §1.16 — CC BY-SA 4.0

## 0. Set up before you start

Same as Lab 1: Python 3.10+ or Colab, and `OPENROUTER_API_KEY` in Colab Secrets or a `.env` file
(how to set up the key, and why we chose this model and provider, are in Lab 1)

In [ ]:
%pip install -q "openai>=1.50" python-dotenv "pydantic>=2.7"

In [ ]:
import os
from collections import Counter
from openai import OpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())
MODEL = "google/gemma-4-26b-a4b-it"
PROVIDER = {"order": ["parasail/bf16"], "allow_fallbacks": False}
EXTRA = {"reasoning": {"enabled": False}, "provider": PROVIDER}

## 1. Asking several times + token probabilities

Ask the same question 5 times at `temperature=1.0` (so the model really samples) and request `logprobs` too
- **how many agree**: if they don't, the model is unsure
- **average log-prob**: near 0 = confident · very negative = unsure

In [ ]:
def sample(question, n=5):
    answers, logprobs = [], []
    for _ in range(n):
        resp = client.chat.completions.create(
            model=MODEL, messages=[{"role": "user", "content": question}],
            max_tokens=30, temperature=1.0, logprobs=True, extra_body=EXTRA)
        answers.append(resp.choices[0].message.content.strip().replace(",", ""))
        tokens = resp.choices[0].logprobs.content
        logprobs.append(sum(t.logprob for t in tokens) / len(tokens))
    return answers, sum(logprobs) / n

In [ ]:
def check(question, n=5):
    answers, avg_lp = sample(question, n)
    top, count = Counter(answers).most_common(1)[0]
    flag = "⚠️ unsure" if count < n * 0.8 or avg_lp < -0.1 else "confident"
    print(f"agree {count}/{n} · logprob {avg_lp:.2f} · {flag} · {answers}")
    return top

## 2. Four questions

| Question | Right answer |
|---|---|
| capital of Australia | Canberra |
| 4738 × 6219 | Python computes it |
| 7851 × 2144 | Python computes it |
| where was the paper *Temporal Patch Attention for Thai Traffic Forecasting* published | **there is no such paper**: we made the title up |

In [ ]:
capital = check("What is the capital of Australia? Answer with the city name only.")

In [ ]:
m1 = check("4738 × 6219 = ? Answer with the number only.")

In [ ]:
m2 = check("7851 × 2144 = ? Answer with the number only.")
paper = check("Which conference published the paper 'Temporal Patch Attention for Thai Traffic Forecasting'? Answer with the conference abbreviation and year only.")

## 3. The limit: confident ≠ correct

Python can check the multiplications; the paper doesn't exist, since we made the title up, so no conference published it

In [ ]:
for a, b, said in [(4738, 6219, m1), (7851, 2144, m2)]:
    print(f"{a} × {b} = {a * b} · model said {said} ·", "✓" if said == str(a * b) else "✗")

## 4. Grounding: answer from a document, with evidence our code can check

The book recommends combining with outside sources: we attach the **course syllabus** to the question, say to answer only from it,
and ask for a `quote` (a line from the document) **before** the answer; then our code checks that the quote really is in the document

In [ ]:
SYLLABUS = """Course syllabus AI301 Agentic AI · Semester 2/2026
- Quiz 1: Wednesday 18 November 2026 (chapters 1–2)
- Quiz 2: Wednesday 9 December 2026 (chapters 3–4)
- Project due: Friday 15 January 2027, before 23:59
- Late work: minus 10% per day, at most 3 days"""

In [ ]:
from pydantic import BaseModel, ConfigDict, Field

class Grounded(BaseModel):
    model_config = ConfigDict(extra="forbid")
    found: bool = Field(description="whether the document contains the answer to this question")
    quote: str = Field(description="the line from the document used to answer, copied word for word, or '' if found is false")
    answer: str = Field(description="a short answer, or 'I don't know' if found is false")

In [ ]:
GROUND_SYSTEM = ("You are a teaching assistant. Answer only from the document in <doc>. Always fill in all three fields. If the document doesn't say, found is false, quote is '' and answer is 'I don't know'. Never guess. "
                 "The quote must be a whole line copied word for word from the document.")
SCHEMA = {"type": "json_schema",
          "json_schema": {"name": "Grounded", "strict": True, "schema": Grounded.model_json_schema()}}

def ask_doc(question):
    msgs = [{"role": "system", "content": GROUND_SYSTEM},
            {"role": "user", "content": f"<doc>\n{SYLLABUS}\n</doc>\n\nQuestion: {question}"}]
    resp = client.chat.completions.create(model=MODEL, messages=msgs, max_tokens=200, temperature=0,
                                          response_format=SCHEMA, extra_body=EXTRA)
    return Grounded.model_validate_json(resp.choices[0].message.content)

In [ ]:
for q in ["When is quiz 3?", "Which chapters does quiz 2 cover?", "At most how many days late can work be?"]:
    g = ask_doc(q)
    print(q, "→", g.answer)
    if not g.found:
        print("   the document doesn't say → don't answer")
    elif g.quote in SYLLABUS:
        print("   ✓ quote is really in the document:", g.quote)
    else:
        print("   ✗ quote is not in the document:", g.quote)

## Summary

| Question | Agree · logprob | Our label | Truth |
|---|---|---|---|
| capital of Australia | 5/5 · ≈ 0 | confident | ✓ |
| 4738 × 6219 | hardly any repeat · clearly negative | ⚠️ unsure | ✗ |
| 7851 × 2144 | the same number almost every time · ≈ 0 | confident | ✗ confidently wrong |
| a paper that doesn't exist | "AAAI 2024" almost every time · ≈ 0 | confident | ✗ no such paper |
| grounding + quote | "I don't know" when the document doesn't say · quotes verified | | ✓ |

**Takeaway for agents:** whatever can be checked in code (arithmetic, a quote in a document), check in code;
that's why a good agent calls a **tool** (a calculator, a document search) instead of guessing, which we'll see in later episodes

**Exercise:** add a line "Final exam: …" to `SYLLABUS`, then ask a question that needs two lines at once,
e.g. "How many days are there between quiz 2 and the final exam?" Does the quote still pass?

▶ **Next: Lab 4 — LangChain** — rebuild the email sorter from Lab 2 with a framework and compare it with our own code